In [1]:
import os
import shutil
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")
torch.manual_seed(42)
np.random.seed(42)

Usando dispositivo: cpu


In [2]:
df = pd.read_csv('clasificados.csv', sep='|', header=0, names=['texto', 'sentimiento'], dtype=str, keep_default_na=False)

df = df[df['sentimiento'].str.strip().str.lower() != 'sentimiento']
df = df[df['texto'].str.strip().str.lower() != 'texto']
df['texto'] = df['texto'].astype(str).str.strip()
df['sentimiento'] = df['sentimiento'].astype(str).str.strip().str.lower()
df = df[df['texto'].str.len() > 0]

label_map = {'negativo': 'negativo', 'neg': 'negativo', 'neutro': 'neutro', 'neu': 'neutro', 'positivo': 'positivo', 'pos': 'positivo'}
df['sentimiento'] = df['sentimiento'].map(label_map)
df = df.dropna(subset=['sentimiento'])
df = df.drop_duplicates(subset=['texto'], keep='first')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Filas únicas finales: {len(df)}")
print(df['sentimiento'].value_counts())

df.to_csv('clasificados_limpio.csv', sep='|', index=False, header=False)

Filas únicas finales: 1221
sentimiento
negativo    483
positivo    415
neutro      323
Name: count, dtype: int64


In [3]:
df = pd.read_csv('./clasificados_limpio.csv', sep='|', header=None, names=['texto', 'sentimiento'])
df['texto'] = df['texto'].astype(str).str.strip()
df['sentimiento'] = df['sentimiento'].astype(str).str.strip()

label_map = {'NEG': 0, 'NEU': 1, 'POS': 2, 'negativo': 0, 'neutro': 1, 'positivo': 2}
df['label'] = df['sentimiento'].map(label_map)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f"Total para entrenar: {len(df)}")
print(df['label'].value_counts())

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_dataset = Dataset.from_pandas(train_df[['texto', 'label']], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[['texto', 'label']], preserve_index=False)

Total para entrenar: 1221
label
0    483
2    415
1    323
Name: count, dtype: int64


In [4]:
model_checkpoint = "./modelo_original"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    id2label={0: "NEG", 1: "NEU", 2: "POS"},
    label2id={"NEG": 0, "NEU": 1, "POS": 2},
    ignore_mismatched_sizes=True
)

max_len = min(tokenizer.model_max_length, model.config.max_position_embeddings, 128)

def tokenize_function(examples):
    return tokenizer(examples["texto"], truncation=True, max_length=max_len, return_token_type_ids=False)

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["texto"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["texto"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df['label']), y=train_df['label'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Pesos: {class_weights.tolist()}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/976 [00:00<?, ? examples/s]

Map:   0%|          | 0/245 [00:00<?, ? examples/s]

Pesos: [0.8428324460983276, 1.2609819173812866, 0.9799196720123291]


In [5]:
num_epochs = 2
batch_size = 16
steps_per_epoch = int(np.ceil(len(tokenized_train) / batch_size))
total_steps = steps_per_epoch * num_epochs
warmup_steps = int(total_steps * 0.15)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        inputs.pop("token_type_ids", None)
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1_macro, 'f1_weighted': f1_weighted, 'precision': precision, 'recall': recall}

training_args = TrainingArguments(
    output_dir="./robertuito-sentiment-finetuned",    # Ruta que guarda los checkpoints
    learning_rate=1.5e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.05,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,
    dataloader_pin_memory=False,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()

carpeta_modelo = "./mejor_modelo"     # Ruta que guarda el mejor modelo
trainer.save_model(carpeta_modelo)
tokenizer.save_pretrained(carpeta_modelo)
print('fine tunning terminado')

Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Weighted,Precision,Recall
1,No log,0.611173,0.800000,0.786152,0.798480,0.787877,0.785122
2,No log,0.601873,0.808163,0.794175,0.806485,0.796144,0.793154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

fine tunning terminado
